## Dataset Requirements

This notebook uses the This American Life Episode Transcripts dataset, available on Kaggle.

Before running the notebook:

1. Download the dataset from [Kaggle](https://www.kaggle.com/datasets/thedevastator/this-american-life-episode-transcripts).
2. Prepare the cleaned dataset as described in the notebook.
3. Place the resulting `lines_clean.csv` file inside a folder named `data` in the same directory as this notebook.

The original dataset and cleaned CSV file are not included in this repository.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("data/lines_clean.csv")

In [ ]:
print(df.head())

In [ ]:
print(df.columns)

## Dataset dimensions

In [ ]:
print("Rows and columns:", df.shape)

## Missing values

In [ ]:
print(df.isnull().sum())

## Data types

In [ ]:
print(df.dtypes)

## A few longer transcript lines

In [ ]:
df['line_text'].sample(10, random_state=42)

In [ ]:
# Create a copy of the original dataframe
df_ner = df.copy()

# Remove rows with missing transcript text
df_ner = df_ner.dropna(subset=['line_text'])

# Remove rows containing only whitespace
df_ner = df_ner[df_ner['line_text'].str.strip() != '']

# Check the new dataset size
print("Original dataset:", df.shape)
print("NER dataset:", df_ner.shape)

In [ ]:
import sys

print(sys.executable)

## Downloading the Spacy Large model

In [ ]:
import sys
!{sys.executable} -m spacy download en_core_web_lg

## Importing the necessary Libraries

In [ ]:
import spacy
import pandas as pd
import json
from itertools import groupby

## Downloading the Small Spacy Model

In [ ]:
import sys
!{sys.executable} -m spacy download en_core_web_sm

## Load/Verify both models

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")
nlp_lg = spacy.load("en_core_web_lg")

print("Small model:", nlp.meta["name"])
print("Large model:", nlp_lg.meta["name"])

In [ ]:
text = "Microsoft announced a new research centre in Lagos. Dr. Sarah Johnson will lead the project."

doc_sm = nlp(text)
doc_lg = nlp_lg(text)

print("SMALL MODEL:")
for ent in doc_sm.ents:
    print(ent.text, ent.label_)

print("\nLARGE MODEL:")
for ent in doc_lg.ents:
    print(ent.text, ent.label_)

## Test Models on Actual data (5 Samples)

In [ ]:
sample = df_ner['line_text'].sample(5, random_state=42)

for text in sample:
    print("\nTEXT:", text)

    doc_sm = nlp(text)
    doc_lg = nlp_lg(text)

    print("SMALL:", [(ent.text, ent.label_) for ent in doc_sm.ents])
    print("LARGE:", [(ent.text, ent.label_) for ent in doc_lg.ents])

## Checking the number of identical comparisons and differences between models (100 Samples)

In [ ]:
sample_100 = df_ner['line_text'].sample(100, random_state=42)

results = []

for text in sample_100:
    doc_sm = nlp(text)
    doc_lg = nlp_lg(text)

    sm_entities = [(ent.text, ent.label_) for ent in doc_sm.ents]
    lg_entities = [(ent.text, ent.label_) for ent in doc_lg.ents]

    results.append({
        "text": text,
        "small_model": sm_entities,
        "large_model": lg_entities,
        "same_predictions": sm_entities == lg_entities
    })

comparison_df = pd.DataFrame(results)

print("Identical predictions:",
      comparison_df['same_predictions'].sum(), "out of 100")

print("Different predictions:",
      (~comparison_df['same_predictions']).sum(), "out of 100")

## Comparing differences between models (10 differences)

In [ ]:
different_predictions = comparison_df[
    comparison_df['same_predictions'] == False
]

for _, row in different_predictions.head(10).iterrows():
    print("\nTEXT:", row['text'])
    print("SMALL:", row['small_model'])
    print("LARGE:", row['large_model'])
    print("-" * 80)

## Calculating number of entities identified by each model

In [ ]:
print("Small model entities:",
      sum(len(x) for x in comparison_df['small_model']))

print("Large model entities:",
      sum(len(x) for x in comparison_df['large_model']))

## Comparing entity categories identified by each model

In [ ]:
from collections import Counter

small_labels = Counter(
    label
    for entities in comparison_df['small_model']
    for _, label in entities
)

large_labels = Counter(
    label
    for entities in comparison_df['large_model']
    for _, label in entities
)

label_comparison = pd.DataFrame({
    'Small Model': small_labels,
    'Large Model': large_labels
}).fillna(0).astype(int)

label_comparison['Difference'] = (
    label_comparison['Small Model'] -
    label_comparison['Large Model']
)

print(label_comparison.sort_index())

## Comparing Differences in ORG category between both Models

In [ ]:
for _, row in comparison_df.iterrows():
    small_org = [
        ent for ent in row['small_model']
        if ent[1] == 'ORG'
    ]

    large_org = [
        ent for ent in row['large_model']
        if ent[1] == 'ORG'
    ]

    if small_org != large_org:
        print("\nTEXT:", row['text'])
        print("SMALL ORG:", small_org)
        print("LARGE ORG:", large_org)
        print("-" * 80)

## Code to Understand labels used in Spacy

In [ ]:
print(spacy.explain("GPE"))
print(spacy.explain("NORP"))
print(spacy.explain("FAC"))
print(spacy.explain("WORK_OF_ART"))

## Create Model Dictionary

In [ ]:
models = {
    'en_core_web_sm': nlp,
    'en_core_web_lg': nlp_lg
}

## Create doc_to_span() function

In [ ]:
def doc_to_spans(doc):
    results = []
    entities = set()

    for ent in doc.ents:
        results.append({
            'from_name': 'label',
            'to_name': 'text',
            'type': 'labels',
            'value': {
                'start': ent.start_char,
                'end': ent.end_char,
                'text': ent.text,
                'labels': [ent.label_]
            }
        })

        entities.add(ent.label_)

    return results, entities

## Test the function

In [ ]:
test_text = "Microsoft announced a new research centre in Lagos. Dr. Sarah Johnson will lead the project."

doc = nlp_lg(test_text)

spans, entities = doc_to_spans(doc)

print("ANNOTATIONS:")
print(json.dumps(spans, indent=2))

print("\nENTITY CATEGORIES:")
print(entities)

## Filter the podcast dataset

In [ ]:
df_easter = df_ner[
    df_ner['line_text'].str.contains("Easter ", na=False)
].copy()

print("Filtered dataset shape:", df_easter.shape)
print(df_easter.head())

## Generate predictions from both models

In [ ]:
entities = set()
tasks = []

for text in df_easter['line_text']:

    predictions = []

    for model_name, nlp_model in models.items():

        doc = nlp_model(text)

        spans, ents = doc_to_spans(doc)

        entities |= ents

        predictions.append({
            'model_version': model_name,
            'result': spans
        })

    tasks.append({
        'data': {'text': text},
        'predictions': predictions
    })

## Check what we've generated

In [ ]:
print("Number of tasks:", len(tasks))
print("Entity categories:", sorted(entities))

print("\nFirst task:")
print(json.dumps(tasks[0], indent=2))

## Save the 2 files

In [ ]:
# Save Label Studio tasks
print(f"Saving {len(tasks)} tasks to tasks.json")

with open('tasks.json', mode='w', encoding='utf-8') as f:
    json.dump(tasks, f, indent=2, ensure_ascii=False)


# Save entity categories
print('Saving entity categories to named_entities.txt')

with open('named_entities.txt', mode='w', encoding='utf-8') as f:
    f.write('\n'.join(sorted(entities)))

## Verify that the files were created

In [ ]:
import os

print("Current working directory:", os.getcwd())

print("\nFiles created:")
print("tasks.json:", os.path.exists("tasks.json"))
print("named_entities.txt:", os.path.exists("named_entities.txt"))